# 02 · Análise — métodos estatísticos da aula aplicados aos dados reais

**Entrada:** `data/processed/*_1t.csv` (gerados no `01_ETL.ipynb`) · **Saída:** tabelas em `data/processed/` e textos em `reports/`.

| Pergunta | Método |
|---|---|
| Votos por candidato a deputado | Mediana, desvio padrão e IQR |
| Partido mais votado por UF | Moda |
| Candidatos que andam juntos entre as UFs | Correlação de Spearman |
| Perfil de votação das 27 UFs | Padronização + PCA |

Regra de ouro da aula: **olhe os dados (histograma) antes de escolher a medida** — os histogramas estão no `03_Visualizacao.ipynb`.

## 0. Configuração e leitura dos dados

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# --- Pastas do repositório: os notebooks ficam em notebooks/, o resto é irmão dessa pasta ---
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW = ROOT / "data" / "raw"              # JSONs originais do TSE (não versionado, ver .gitignore)
PROC = ROOT / "data" / "processed"       # CSVs gerados pelo ETL e pelas análises (versionado)
FIG = ROOT / "reports" / "figures"       # gráficos
REP = ROOT / "reports"                   # textos entregáveis
for d in (RAW, PROC, FIG, REP):
    d.mkdir(parents=True, exist_ok=True)

REGIAO = {**dict.fromkeys(["AC", "AP", "AM", "PA", "RO", "RR", "TO"], "Norte"),
          **dict.fromkeys(["AL", "BA", "CE", "MA", "PB", "PE", "PI", "RN", "SE"], "Nordeste"),
          **dict.fromkeys(["DF", "GO", "MT", "MS"], "Centro-Oeste"),
          **dict.fromkeys(["ES", "MG", "RJ", "SP"], "Sudeste"),
          **dict.fromkeys(["PR", "RS", "SC"], "Sul")}

ARQUIVOS = {"presidente": "presidente_1t.csv", "senador": "senador_1t.csv",
            "deputado_federal": "deputado_federal_1t.csv", "deputado_estadual": "deputado_estadual_1t.csv"}
df = pd.concat([pd.read_csv(PROC / nome) for nome in ARQUIVOS.values()], ignore_index=True)
dfv = df[df["situacao_candidatura"].eq("Válido")]     # base das análises: votos válidos (sem "Anulado sub judice")
print(f"{len(df)} linhas | {df['cargo'].nunique()} cargos")
df.groupby("cargo")["uf"].nunique().rename("unidades").to_frame().T

18998 linhas | 4 cargos


cargo,deputado_estadual,deputado_federal,presidente,senador
unidades,27,27,29,27


## 1. Votos por candidato a deputado — tendência central, dispersão e outliers

In [2]:
def limites_tukey(x):
    """Quartis, IQR e limite superior de Tukey (Q3 + 1,5 × IQR)."""
    q1, q3 = np.percentile(x, [25, 75])
    return q1, q3, q3 - q1, q3 + 1.5 * (q3 - q1)


def resumo(x):
    q1, q3, iqr, lim = limites_tukey(x)
    return pd.Series({
        "candidatos": len(x), "média": x.mean(), "mediana": x.median(), "desvio_padrão": x.std(),   # pandas: ÷ (n−1)
        "CV_%": x.std() / x.mean() * 100, "Q1": q1, "Q3": q3, "IQR": iqr, "limite_Tukey": lim,
        "outliers": int((x > lim).sum()), "outliers_%": (x > lim).mean() * 100, "máximo": x.max()})


estat = pd.DataFrame({c: resumo(dfv.loc[dfv["cargo"] == c, "votos"])
                      for c in ["deputado_federal", "deputado_estadual"]})
estat.to_csv(PROC / "analise_estatisticas_votos_por_candidato.csv", encoding="utf-8-sig")
estat.round(1)

,deputado_federal,deputado_estadual
candidatos,7051.0,10847.0
média,15635.5,9723.3
mediana,1270.0,813.0
desvio_padrão,67459.8,30314.6
CV_%,431.5,311.8
Q1,368.0,229.0
Q3,6453.5,6318.0
IQR,6085.5,6089.0
limite_Tukey,15581.8,15451.5
outliers,1192.0,1824.0


In [3]:
# Padronização (z-score) como segunda régua de outliers: |z| > 3 (regra prática da aula)
for cargo in ["deputado_federal", "deputado_estadual"]:
    x = dfv.loc[dfv["cargo"] == cargo, "votos"]
    z = (x - x.mean()) / x.std()
    print(f"{cargo:18s} Tukey: {int((x > limites_tukey(x)[3]).sum()):5d} outliers | |z| > 3: {int((z.abs() > 3).sum()):4d} candidatos "
          f"| média/mediana = {x.mean() / x.median():.1f}")

# Outliers de Tukey são erros? Quantos eleitos estão acima do limite (deputado federal)?
dep = dfv[dfv["cargo"] == "deputado_federal"]
eleitos_fed = dep[dep["resultado"].str.startswith("Eleito")]
lim_fed = limites_tukey(dep["votos"])[3]
print(f"\nDeputado Federal: {int((eleitos_fed['votos'] > lim_fed).sum())} dos {len(eleitos_fed)} eleitos com resultado final "
      f"estão acima do limite de Tukey ({lim_fed:,.0f} votos)".replace(",", "."))

# Tabela por UF (deputado federal): mediana, desvio padrão e IQR dentro de cada estado
por_uf = (dep.groupby("uf")["votos"]
          .agg(candidatos="count", mediana="median", média="mean", desvio_padrão="std",
               IQR=lambda s: np.subtract(*np.percentile(s, [75, 25])), máximo="max").round(0))
por_uf.to_csv(PROC / "analise_deputado_federal_por_uf.csv", encoding="utf-8-sig")
por_uf.sort_values("mediana", ascending=False).head(8)

deputado_federal   Tukey:  1192 outliers | |z| > 3:   41 candidatos | média/mediana = 12.3
deputado_estadual  Tukey:  1824 outliers | |z| > 3:   98 candidatos | média/mediana = 12.0

Deputado Federal: 502 dos 505 eleitos com resultado final estão acima do limite de Tukey (15.582 votos)


,candidatos,mediana,média,desvio_padrão,IQR,máximo
uf,,,,,,
MA,264,2202.0,14610.0,34560.0,4820.0,188853
MT,127,1900.0,14512.0,31054.0,12287.0,205064
SC,218,1897.0,18850.0,48974.0,14111.0,560223
SP,983,1841.0,23386.0,119837.0,9250.0,3038438
PA,247,1825.0,19048.0,50411.0,8320.0,414927
ES,130,1746.0,15961.0,44305.0,9074.0,419735
RO,121,1677.0,7471.0,17165.0,7087.0,158421
RS,411,1624.0,14368.0,39429.0,4572.0,379834


## 2. Partido mais votado por UF — moda

In [4]:
# Partido mais votado = o que soma mais votos nominais na UF. A "moda" estatística (df["partido"].mode())
# é o partido mais frequente: aqui, entre os deputados federais ELEITOS de cada UF.
mais_votado = dep.groupby(["uf", "partido"])["votos"].sum().reset_index().sort_values("votos", ascending=False)
mais_votado = mais_votado.drop_duplicates("uf").set_index("uf").rename(
    columns={"partido": "partido_mais_votado", "votos": "votos_do_partido"})
eleitos = dep[dep["resultado"].str.startswith("Eleito")]
moda_eleitos = eleitos.groupby("uf")["partido"].agg(lambda s: ", ".join(s.mode())).rename("moda_partido_entre_eleitos")
qtd = eleitos.groupby("uf").size().rename("eleitos")
moda_uf = mais_votado.join(moda_eleitos).join(qtd).sort_index()
moda_uf["moda_partido_entre_eleitos"] = moda_uf["moda_partido_entre_eleitos"].fillna("sem totalização final")   # RO
moda_uf["eleitos"] = moda_uf["eleitos"].fillna(0).astype(int)
moda_uf.to_csv(PROC / "analise_moda_partido_por_uf.csv", encoding="utf-8-sig")

print("Partido mais votado em nº de UFs:")
print(moda_uf["partido_mais_votado"].value_counts().to_string())
print("\nModa nacional do partido do candidato (partido com mais candidatos):", list(dep["partido"].mode()))
moda_uf

Partido mais votado em nº de UFs:
partido_mais_votado
PL              13
MDB              3
REPUBLICANOS     3
PP               2
PT               2
UNIÃO            1
PSB              1
NOVO             1
PSD              1

Moda nacional do partido do candidato (partido com mais candidatos): ['PL']


,partido_mais_votado,votos_do_partido,moda_partido_entre_eleitos,eleitos
uf,,,,
AC,PP,89075,"PP, UNIÃO",8
AL,PP,502180,PP,9
AM,MDB,391195,"MDB, PL, UNIÃO",8
AP,UNIÃO,101120,UNIÃO,8
BA,PT,1361897,PT,39
CE,PL,1089187,"PL, PSB",22
DF,PL,436068,PL,8
ES,PL,527677,"PL, PT, UNIÃO",10
GO,PL,619989,PL,17


## 3. Candidatos que andam juntos entre as UFs — correlação de Spearman

Usamos o **presidente**: para cada candidato, o percentual de votos válidos nas 27 UFs. Spearman (sobre os postos) é a
escolha segura porque os percentuais são assimétricos e há UFs atípicas; comparamos com Pearson.

In [5]:
pct = (dfv[(dfv["cargo"] == "presidente") & (~dfv["uf"].isin(["BR", "ZZ"]))]
       .pivot(index="uf", columns="candidato", values="pct_validos"))          # linhas: 27 UFs · colunas: candidatos
assert pct.shape[0] == 27 and pct.notna().all().all()
print(pct.shape, "-> 27 UFs ×", pct.shape[1], "candidatos")

spear = pct.corr(method="spearman")
pears = pct.corr(method="pearson")
spear.to_csv(PROC / "analise_spearman_presidente.csv", encoding="utf-8-sig")

# Pares mais correlacionados (positivos e negativos), com Pearson ao lado para comparar
pares = (spear.rename_axis(index="candidato_A", columns="candidato_B")
         .where(np.triu(np.ones(spear.shape), 1).astype(bool)).stack().dropna().rename("spearman").reset_index())
pares["pearson"] = [pears.loc[a, b] for a, b in zip(pares.candidato_A, pares.candidato_B)]
pares = pares.sort_values("spearman")
pd.concat([pares.head(4), pares.tail(4)]).round(2).reset_index(drop=True)

(27, 12) -> 27 UFs × 12 candidatos


,candidato_A,candidato_B,spearman,pearson
0,FLAVIO BOLSONARO,LULA,-0.99,-0.99
1,LULA,SAMARA,-0.56,-0.50
2,CLARIANA BARAO,LULA,-0.55,-0.54
3,LULA,RENAN SANTOS,-0.53,-0.53
4,RENAN SANTOS,VETERINÁRIO WILSON GRASSI,0.74,0.74
5,RENAN SANTOS,RUI COSTA PIMENTA,0.80,0.82
6,SAMARA,VETERINÁRIO WILSON GRASSI,0.87,0.90
7,RENAN SANTOS,SAMARA,0.89,0.86


## 4. Perfil de votação das 27 UFs — padronização + PCA (passo a passo)

Seguindo os 5 passos da aula, com a matriz **linhas = 27 UFs, colunas = candidatos a presidente** (% de votos válidos):

1. monta a matriz (`pct`, acima) → 2. padroniza cada coluna (z-score) → 3. ajusta o PCA e lê a variância explicada →
4. escolhe *k* (80–90 % acumulado ou cotovelo) → 5. interpreta os loadings (o gráfico PC1 × PC2 está no `03_Visualizacao.ipynb`).

In [6]:
# Passo 2: padronização (média 0, desvio 1 em cada coluna). Obrigatória antes do PCA.
Z = StandardScaler().fit_transform(pct)

# Passo 3: PCA completo e variância explicada
pca = PCA().fit(Z)
var = pd.DataFrame({"componente": [f"PC{i+1}" for i in range(len(pca.explained_variance_ratio_))],
                    "variancia_explicada_%": pca.explained_variance_ratio_ * 100})
var["acumulada_%"] = var["variancia_explicada_%"].cumsum()
var.round(2).to_csv(PROC / "pca_variancia.csv", index=False, encoding="utf-8-sig")
print(var.round(1).head(6).to_string(index=False))

# Passo 4: escolha de k
k80 = int((var["acumulada_%"] < 80).sum() + 1)
print(f"\nPrimeiro k com >= 80% acumulado: k = {k80}  |  PC1+PC2 = {var['acumulada_%'].iloc[1]:.1f}%")

componente  variancia_explicada_%  acumulada_%
       PC1                   49.6         49.6
       PC2                   17.4         67.0
       PC3                   11.3         78.3
       PC4                    6.1         84.4
       PC5                    5.5         89.9
       PC6                    3.9         93.8

Primeiro k com >= 80% acumulado: k = 4  |  PC1+PC2 = 67.0%


In [7]:
# Contraste com a aula: o que acontece SEM padronizar?
pca_sem = PCA().fit(pct.values)
print(f"PC1 com dados padronizados : {pca.explained_variance_ratio_[0] * 100:5.1f}% da variância")
print(f"PC1 SEM padronizar         : {pca_sem.explained_variance_ratio_[0] * 100:5.1f}% da variância "
      "(reflete a escala: candidatos grandes dominam)")

PC1 com dados padronizados :  49.6% da variância
PC1 SEM padronizar         :  98.3% da variância (reflete a escala: candidatos grandes dominam)


In [8]:
# Passo 5: loadings (peso de cada candidato em cada componente) e scores (posição de cada UF)
loadings = pd.DataFrame(pca.components_[:3].T, index=pct.columns, columns=["PC1", "PC2", "PC3"])
loadings.index.name = "candidato"
loadings.round(3).to_csv(PROC / "pca_loadings.csv", encoding="utf-8-sig")
scores = pd.DataFrame(pca.transform(Z)[:, :2], index=pct.index, columns=["PC1", "PC2"])
scores["regiao"] = scores.index.map(REGIAO)
scores.round(3).to_csv(PROC / "pca_scores_ufs.csv", encoding="utf-8-sig")

partidos = dfv[(dfv["cargo"] == "presidente") & (dfv["uf"] == "BR")].set_index("candidato")["partido"]
loadings.round(2).join(partidos.rename("partido")).sort_values("PC1")

,PC1,PC2,PC3,partido
candidato,,,,
LULA,-0.25,-0.53,-0.02,PT
RONALDO CAIADO,0.02,0.27,0.72,PSD
CLARIANA BARAO,0.17,0.36,-0.44,DC
ESCRITOR AUGUSTO CURY,0.23,-0.24,-0.44,AVANTE
FLAVIO BOLSONARO,0.24,0.54,-0.08,PL
EDMILSON COSTA,0.27,-0.02,0.04,PCB
ZEMA,0.27,-0.11,0.09,NOVO
HERTZ DIAS,0.31,-0.35,0.17,PSTU
RUI COSTA PIMENTA,0.36,-0.14,0.17,PCO


### Leitura dos loadings

**Variância explicada:** PC1 = 49,6 %, PC2 = 17,4 % (acumulado 67,0 %), PC3 = 11,3 % (78,3 %); só com 4 componentes passa de 80 % (84,4 %). Ou seja, dois eixos resumem **dois terços** do perfil de votação — menos que no exemplo sintético da aula (95 %), porque aqui os candidatos não são tão correlacionados entre si. Sem padronizar, o PC1 "explicaria" 98,3 %: só refletiria a escala (Lula e Flávio têm dezenas de pontos percentuais; os demais, décimos).

**PC1 (49,6 %) — peso dos candidatos menores (votação pulverizada).** Todos os candidatos menores têm peso positivo (+0,23 a +0,39: Samara 0,39, Renan Santos 0,37, Wilson Grassi 0,37, Rui Costa Pimenta 0,36…), Lula tem peso negativo (−0,25) e Caiado fica perto de zero (0,02). Na prática, o escore do PC1 acompanha a **soma dos percentuais dos nove candidatos menores** (correlação de Spearman de 0,89 com essa soma) e, em menor grau, o voto em Lula (−0,65). PC1 alto: SP (6,01; os menores somam 7,5 %), RJ (4,24), DF (3,73), SC (2,86), PR (2,36), MG (2,30), RS (2,25) — Sul, Sudeste e DF, onde os menores somam de 5,4 % a 7,5 % (mediana nacional: 4,9 %). PC1 muito negativo: PI (−2,97), PE (−2,90), BA (−2,86) e MA (−2,76), onde os menores somam cerca de 4 % e Lula tem 63 % a 71 %; e TO (−2,74), que **não** segue o padrão de Lula alto (43,4 %): seu PC1 baixo vem de ter poucos votos nos candidatos menores (4,0 %).

**PC2 (17,4 %) — Flávio Bolsonaro × Lula, descontado o PC1.** Pesos positivos: Flávio (0,54), Clariana Barão (0,36), Caiado (0,27); negativos: Lula (−0,53), Hertz Dias (−0,35), Augusto Cury (−0,24). UFs altas: MT (3,52), GO (2,83), RR (2,35), RO (2,04), AC (1,99) — **Centro-Oeste e Norte**, com votação muito forte em Flávio (ex.: MT 65,1 % contra 29,2 % de Lula). UFs baixas: o Nordeste e SP/MG.

**PC3 (11,3 %)** é quase um eixo só do **Ronaldo Caiado** (0,72): ele teve 12,3 % em GO e no máximo 4,8 % nas demais UFs. Tem sentido: é um efeito de estado de origem, que não aparece em PC1/PC2.

**Cuidados:** (i) são só 27 observações e 12 variáveis; (ii) os percentuais somam 100 % por UF (dados composicionais), então Lula e Flávio (92,9 % dos votos válidos em média) são quase espelhos um do outro — por isso ρ(Lula, Flávio) = −0,99 e eles pesam em lados opostos; (iii) os componentes são combinações, não variáveis originais; (iv) o PCA descreve **associação**, não causa: o gráfico não diz *por que* o Nordeste votou assim.

## 5. Interpretação dos resultados

**Votos por candidato a deputado (mediana, desvio padrão, IQR).** A distribuição é extremamente assimétrica. Para Deputado Federal, a **mediana é 1.270 votos** e a **média, 15.636** (12,3 vezes maior); o desvio padrão (67.460) dá um **CV de 432 %**, e o IQR é 6.086 (Q1 = 368; Q3 = 6.454). Para Deputado Estadual/Distrital: mediana 813, média 9.723, CV 312 %. A média **não representa** o candidato típico; a mediana e o IQR sim. Pela regra de Tukey (limite ≈ 15,6 mil votos no federal), 1.192 candidatos (16,9 %) são outliers — e **502 dos 505 eleitos com resultado final estão acima do limite**: aqui o outlier não é erro, é o candidato bem votado. O critério |z| > 3 é bem mais restritivo (41 candidatos no federal), porque a própria média e o desvio são inflados pela cauda.

**Partido mais votado por UF (moda).** Para Deputado Federal, o **PL** foi o partido mais votado em **13 das 27 UFs**; MDB e Republicanos em 3 cada; PP e PT em 2; UNIÃO, PSB, NOVO e PSD em 1 cada. O PL também é a moda nacional (partido com mais candidatos). Em Rondônia não há moda entre eleitos porque o arquivo do TSE veio sem a totalização final (sem a situação dos candidatos).

**Correlação de Spearman entre candidatos a presidente (27 UFs).** ρ(Flávio, Lula) = **−0,99**: onde um vai bem, o outro vai mal (reforçado pela soma fixa de 100 %). Entre os candidatos menores as correlações são **positivas e altas**: Renan Santos × Samara 0,89, Samara × Wilson Grassi 0,87, Renan × Rui Costa Pimenta 0,80 — eles tendem a ir bem (ou mal) nas mesmas UFs. Pearson e Spearman ficaram praticamente iguais (por exemplo, 0,89 e 0,86), então não há relação fortemente não linear nem outlier dominando esses pares. Correlação não é causalidade.

**Padronização + PCA.** Ver a leitura dos loadings acima: dois eixos (PC1 = peso dos candidatos menores, baixo no Nordeste e alto em SP, RJ, DF e no Sul; PC2 = Flávio × Lula, ligado a Centro-Oeste/Norte × Nordeste) resumem 67 % do perfil de votação das UFs.

## 6. Limites e cuidados (parágrafo de 5 a 8 linhas)

Os dados são um retrato dos arquivos públicos do TSE, coletados em 16/10/2026, e podem mudar com o julgamento de candidaturas: votos anulados sub judice ficaram de fora e, para os deputados de Rondônia, o arquivo do TSE veio sem totalização final (embora a imprensa já divulgasse os eleitos), então essa UF não tem eleitos nos CSVs. No Senado cada eleitor tem dois votos, logo os percentuais são sobre votos, não sobre eleitores. A distribuição de votos por candidato é muito assimétrica, por isso priorizamos mediana e IQR; os outliers não são erros, pois incluem 502 dos 505 eleitos com resultado final. O PCA usa só 27 UFs e percentuais que somam 100 %; dois eixos explicam 67 % da variância, e os padrões mostram associação, não causa.

In [9]:
# Grava os textos entregáveis (os mesmos das seções acima) em reports/
PARAGRAFO = 'Os dados são um retrato dos arquivos públicos do TSE, coletados em 16/10/2026, e podem mudar com o julgamento de candidaturas: votos anulados sub judice ficaram de fora e, para os deputados de Rondônia, o arquivo do TSE veio sem totalização final (embora a imprensa já divulgasse os eleitos), então essa UF não tem eleitos nos CSVs. No Senado cada eleitor tem dois votos, logo os percentuais são sobre votos, não sobre eleitores. A distribuição de votos por candidato é muito assimétrica, por isso priorizamos mediana e IQR; os outliers não são erros, pois incluem 502 dos 505 eleitos com resultado final. O PCA usa só 27 UFs e percentuais que somam 100 %; dois eixos explicam 67 % da variância, e os padrões mostram associação, não causa.'
LEITURA_PCA = '**Variância explicada:** PC1 = 49,6 %, PC2 = 17,4 % (acumulado 67,0 %), PC3 = 11,3 % (78,3 %); só com 4 componentes passa de 80 % (84,4 %). Ou seja, dois eixos resumem **dois terços** do perfil de votação — menos que no exemplo sintético da aula (95 %), porque aqui os candidatos não são tão correlacionados entre si. Sem padronizar, o PC1 "explicaria" 98,3 %: só refletiria a escala (Lula e Flávio têm dezenas de pontos percentuais; os demais, décimos).\n\n**PC1 (49,6 %) — peso dos candidatos menores (votação pulverizada).** Todos os candidatos menores têm peso positivo (+0,23 a +0,39: Samara 0,39, Renan Santos 0,37, Wilson Grassi 0,37, Rui Costa Pimenta 0,36…), Lula tem peso negativo (−0,25) e Caiado fica perto de zero (0,02). Na prática, o escore do PC1 acompanha a **soma dos percentuais dos nove candidatos menores** (correlação de Spearman de 0,89 com essa soma) e, em menor grau, o voto em Lula (−0,65). PC1 alto: SP (6,01; os menores somam 7,5 %), RJ (4,24), DF (3,73), SC (2,86), PR (2,36), MG (2,30), RS (2,25) — Sul, Sudeste e DF, onde os menores somam de 5,4 % a 7,5 % (mediana nacional: 4,9 %). PC1 muito negativo: PI (−2,97), PE (−2,90), BA (−2,86) e MA (−2,76), onde os menores somam cerca de 4 % e Lula tem 63 % a 71 %; e TO (−2,74), que **não** segue o padrão de Lula alto (43,4 %): seu PC1 baixo vem de ter poucos votos nos candidatos menores (4,0 %).\n\n**PC2 (17,4 %) — Flávio Bolsonaro × Lula, descontado o PC1.** Pesos positivos: Flávio (0,54), Clariana Barão (0,36), Caiado (0,27); negativos: Lula (−0,53), Hertz Dias (−0,35), Augusto Cury (−0,24). UFs altas: MT (3,52), GO (2,83), RR (2,35), RO (2,04), AC (1,99) — **Centro-Oeste e Norte**, com votação muito forte em Flávio (ex.: MT 65,1 % contra 29,2 % de Lula). UFs baixas: o Nordeste e SP/MG.\n\n**PC3 (11,3 %)** é quase um eixo só do **Ronaldo Caiado** (0,72): ele teve 12,3 % em GO e no máximo 4,8 % nas demais UFs. Tem sentido: é um efeito de estado de origem, que não aparece em PC1/PC2.\n\n**Cuidados:** (i) são só 27 observações e 12 variáveis; (ii) os percentuais somam 100 % por UF (dados composicionais), então Lula e Flávio (92,9 % dos votos válidos em média) são quase espelhos um do outro — por isso ρ(Lula, Flávio) = −0,99 e eles pesam em lados opostos; (iii) os componentes são combinações, não variáveis originais; (iv) o PCA descreve **associação**, não causa: o gráfico não diz *por que* o Nordeste votou assim.'

(REP / "paragrafo_limites_e_cuidados.md").write_text(PARAGRAFO + "\n", encoding="utf-8")
(REP / "leitura_loadings_pca.md").write_text(LEITURA_PCA + "\n", encoding="utf-8")
print("Parágrafo:", len(PARAGRAFO.split()), "palavras ->", (REP / "paragrafo_limites_e_cuidados.md").relative_to(ROOT))
print("Leitura do PCA ->", (REP / "leitura_loadings_pca.md").relative_to(ROOT))

Parágrafo: 128 palavras -> reports/paragrafo_limites_e_cuidados.md
Leitura do PCA -> reports/leitura_loadings_pca.md


**Próximo passo:** abra `03_Visualizacao.ipynb`.